# House Price Prediction Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

# Set up paths
base_dir = Path.cwd()
print(f"Working directory: {base_dir}")

In [ ]:
# Load dataset
possible_paths = [
    base_dir / "house_price.csv",
    base_dir / "House_Price_Data.xlsx",
    base_dir / "House Prediction data set.csv",
    base_dir / "House Prediction data set.xlsx",
]
possible_paths += list(base_dir.glob("*house*price*.csv"))
possible_paths += list(base_dir.glob("*house*price*.xlsx"))

csv_path = None
for path in possible_paths:
    if path.exists():
        csv_path = path
        break

if csv_path is None:
    raise FileNotFoundError("No dataset found")

if csv_path.suffix.lower() == ".csv":
    df = pd.read_csv(csv_path)
else:
    df = pd.read_excel(csv_path)

print(f"Dataset loaded from: {csv_path}")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print(df.info())
print("\n")
print(df.describe())
print("\n")
print("Missing values:")
print(df.isnull().sum())

In [ ]:
# Data preprocessing
num_cols = df.select_dtypes(include=np.number).columns
for col in num_cols:
    df.loc[:, col] = df[col].fillna(df[col].median())

cat_cols = df.select_dtypes(include=["object", "string"]).columns
for col in cat_cols:
    if not df[col].mode().empty:
        df.loc[:, col] = df[col].fillna(df[col].mode()[0])

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print("Data preprocessing complete")

## Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# Prepare features and target
if 'price' not in df.columns:
    raise KeyError(f"Expected target column 'price' not found")

df['log_price'] = np.log1p(df['price'])
X = df.drop(['price', 'log_price'], axis=1)
y = df['log_price']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Train-test split and scaling
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

In [ ]:
# Train models
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

rf = RandomForestRegressor()
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

gb = GradientBoostingRegressor()
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)

print("All models trained successfully!")

## Model Evaluation Results

In [ ]:
def evaluate_model(name, y_test, pred):
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    print(f"{name} Results")
    print("-"*30)
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R2 Score:", r2)
    print()

evaluate_model("Linear Regression", y_test, lr_pred)
evaluate_model("Random Forest", y_test, rf_pred)
evaluate_model("Gradient Boosting", y_test, gb_pred)

## Residual Analysis

In [ ]:
residuals = y_test - gb_pred

plt.figure(figsize=(8, 6))
sns.scatterplot(x=gb_pred, y=residuals)
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residual Analysis")
plt.tight_layout()
plt.show()

## Predicted vs Actual (Gradient Boosting)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test, gb_pred, alpha=0.5, s=50)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel("Actual Log-Price")
plt.ylabel("Predicted Log-Price")
plt.title("Predicted vs Actual (Gradient Boosting)")
plt.legend()
plt.tight_layout()
plt.show()

## Feature Importance - Random Forest

In [ ]:
rf_importances = rf.feature_importances_
feature_names = X.columns
indices = np.argsort(rf_importances)[-15:]  # Top 15 features

plt.figure(figsize=(10, 6))
plt.barh(range(len(indices)), rf_importances[indices])
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Feature Importance")
plt.title("Top 15 Features - Random Forest")
plt.tight_layout()
plt.show()

## Price Distribution - Actual vs Predicted

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(y_test, bins=30, alpha=0.5, label='Actual', edgecolor='black')
plt.hist(gb_pred, bins=30, alpha=0.5, label='Predicted (GB)', edgecolor='black')
plt.xlabel("Log-Price")
plt.ylabel("Frequency")
plt.title("Distribution: Actual vs Predicted Prices (Gradient Boosting)")
plt.legend()
plt.tight_layout()
plt.show()

## Model Comparison

In [ ]:
models = ['Linear Regression', 'Random Forest', 'Gradient Boosting']
r2_scores = [
    r2_score(y_test, lr_pred),
    r2_score(y_test, rf_pred),
    r2_score(y_test, gb_pred)
]
mae_scores = [
    mean_absolute_error(y_test, lr_pred),
    mean_absolute_error(y_test, rf_pred),
    mean_absolute_error(y_test, gb_pred)
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.bar(models, r2_scores, color=['blue', 'green', 'red'])
ax1.set_ylabel("R² Score")
ax1.set_title("R² Score Comparison")
ax1.axhline(y=0, color='k', linestyle='--', alpha=0.3)

ax2.bar(models, mae_scores, color=['blue', 'green', 'red'])
ax2.set_ylabel("MAE")
ax2.set_title("MAE Comparison")

plt.tight_layout()
plt.show()

In [ ]:
# Optional: Save model
model_path = base_dir / 'house_price_model.pkl'
joblib.dump(gb, model_path)
print(f"Model saved to {model_path}")